## Show, Attend and Tell — Training (Flickr8k)

This notebook is the Jupyter version of `code/main.py`, structured for running on a cloud GPU.

- **Model**: ResNet encoder → soft-attention → LSTMCell decoder
- **Training**: teacher forcing + CE loss + doubly-stochastic attention penalty


In [2]:
!git clone https://github.com/seanzhangw/show-attend-tell.git

Cloning into 'show-attend-tell'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [ ]:
%cd /content/show-attend-tell
!git pull

%cd /content/show-attend-tell/

/content/show-attend-tell
Already up to date.
[Errno 2] No such file or directory: '/content/show-attend-tell/code'
/content/show-attend-tell


In [7]:
# =========================
# 1) Imports
# =========================
import time

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms

import config
from datasets.flickr8k import build_flickr8k_dataset, collate_fn
from models.encoder import EncoderCNN
from models.decoder import Decoder


ModuleNotFoundError: No module named 'config'

In [ ]:
# =========================
# 2) Loss helpers
# =========================

def compute_caption_loss(logits, captions, criterion):
    """
    Args:
        logits:   (B, T-1, V)
        captions: (B, T)

    Returns:
        ce_loss: scalar
    """
    targets = captions[:, 1:]  # (B, T-1)
    bsz, time_steps, vocab_size = logits.shape

    logits_flat = logits.reshape(bsz * time_steps, vocab_size)   # (B*(T-1), V)
    targets_flat = targets.reshape(bsz * time_steps)             # (B*(T-1))

    return criterion(logits_flat, targets_flat)


In [ ]:
# =========================
# 3) Train / Validate
# =========================

def train_one_epoch(
    encoder,
    decoder,
    loader,
    criterion,
    optimizer,
    device,
    lambda_att=1.0,
    grad_clip=5.0,
    print_every=100,
    freeze_encoder=True,
):
    """One training epoch. Returns average total loss."""

    if freeze_encoder:
        encoder.eval()
    else:
        encoder.train()
    decoder.train()

    running_loss = 0.0
    t0 = time.time()

    for batch_idx, (images, captions) in enumerate(loader, start=1):
        images = images.to(device)    # (B, 3, 224, 224)
        captions = captions.to(device)  # (B, T)

        optimizer.zero_grad()

        if freeze_encoder:
            with torch.no_grad():
                features = encoder(images)  # (B, L, D)
        else:
            features = encoder(images)      # (B, L, D)

        logits, alphas = decoder(features, captions)  # (B, T-1, V), (B, T-1, L)

        ce_loss = compute_caption_loss(logits, captions, criterion)

        # Doubly stochastic attention penalty:
        # For each location i, sum_t alpha_ti should be close to 1.
        attention_penalty = ((1.0 - alphas.sum(dim=1)) ** 2).mean()

        loss = ce_loss + lambda_att * attention_penalty
        loss.backward()

        if grad_clip is not None:
            clip_grad_norm_(decoder.parameters(), grad_clip)
            if not freeze_encoder:
                clip_grad_norm_(encoder.parameters(), grad_clip)

        optimizer.step()

        running_loss += loss.item()

        # ---- Throughput / ETA (tiny print) ----
        if batch_idx % print_every == 0:
            elapsed = time.time() - t0
            it_per_sec = batch_idx / max(elapsed, 1e-8)
            sec_per_it = 1.0 / max(it_per_sec, 1e-8)
            eta_sec = (len(loader) - batch_idx) * sec_per_it

            avg_so_far = running_loss / batch_idx
            print(
                f"[Train] {batch_idx:>5}/{len(loader)} "
                f"loss={avg_so_far:.4f} (CE={ce_loss.item():.4f}, Att={attention_penalty.item():.4f}) | "
                f"{it_per_sec:.2f} it/s | ETA {eta_sec/60:.1f} min"
            )

    return running_loss / max(1, len(loader))


@torch.no_grad()
def validate(encoder, decoder, loader, criterion, device, lambda_att=1.0):
    """Validation epoch. Returns average total loss."""

    encoder.eval()
    decoder.eval()

    running_loss = 0.0

    for images, captions in loader:
        images = images.to(device)
        captions = captions.to(device)

        features = encoder(images)              # (B, L, D)
        logits, alphas = decoder(features, captions)  # (B, T-1, V), (B, T-1, L)

        ce_loss = compute_caption_loss(logits, captions, criterion)
        attention_penalty = ((1.0 - alphas.sum(dim=1)) ** 2).mean()
        loss = ce_loss + lambda_att * attention_penalty

        running_loss += loss.item()

    return running_loss / max(1, len(loader))


In [ ]:
# =========================
# 4) Setup (data, model, optimizer)
# =========================

# Hyperparameters (edit as needed)
epochs = 2
batch_size = config.BATCH_SIZE
lr = 1e-4
embed_dim = 512
hidden_dim = 512
attention_dim = 512

dropout = 0.5
lambda_att = 1.0
grad_clip = 5.0
freeze_encoder = True
val_ratio = 0.1

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ImageNet normalization for ResNet
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

# Dataset
dataset, word2idx, idx2word = build_flickr8k_dataset(config, transform=transform)
pad_idx = word2idx["<pad>"]
print(f"Dataset size: {len(dataset)} | Vocab size: {len(word2idx)}")

# Train/val split
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

# Models
encoder = EncoderCNN().to(device)
decoder = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# Loss / Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

params = list(decoder.parameters())
if not freeze_encoder:
    params += [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=lr)


In [ ]:
# =========================
# 5) Train
# =========================

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")

    train_loss = train_one_epoch(
        encoder=encoder,
        decoder=decoder,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        lambda_att=lambda_att,
        grad_clip=grad_clip,
        print_every=100,
        freeze_encoder=freeze_encoder,
    )

    val_loss = validate(
        encoder=encoder,
        decoder=decoder,
        loader=val_loader,
        criterion=criterion,
        device=device,
        lambda_att=lambda_att,
    )

    print(f"[Epoch {epoch}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")


**Testing**


In [11]:
# Imports
from torch.utils.data import DataLoader
from datasets.flickr8k import build_flickr8k_dataset, collate_fn

ModuleNotFoundError: No module named 'datasets.flickr8k'

In [ ]:

# Build dataset + vocab
dataset, word2idx, idx2word = build_flickr8k_dataset(config)

print(f"Dataset size: {len(dataset)}")
print(f"Vocab size: {len(word2idx)}")

# Create DataLoader
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

# Inspect one batch
for images, captions in loader:
    print("Image batch shape:", images.shape)      # (B, 3, H, W)
    print("Caption batch shape:", captions.shape)  # (B, max_len)
    
    print("\nSample caption (token IDs):")
    print(captions[0])
    
    break